# 38장 · 재정렬과 다중 질의 검색

재정렬은 기존 후보들의 순서를 바꾸고, 다중 질의 검색은 질문 표현을 늘려 후보를 모읍니다. 첫 검색에 정답 문서가 없는 경우 두 방법의 차이가 드러납니다.

대출 기간을 묻는 질문에서 마침표만 담은 청크가 1위가 됐습니다. 기존 후보를 다시 정렬하는 방법과 질의를 확장해 새 후보를 가져오는 방법을 같은 자료에서 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## sorted()를 이용한 후보 재정렬

37장의 16글자 청크 중 library-1:0에는 14일이 있고 library-1:1에는 마침표만 있습니다. 순위 차이를 눈으로 보기 위해 이 두 입력의 순서를 뒤집어 둡니다. 여기서 쓰는 점수는 수업용 단어 일치이며 학습된 재순위 모델의 결과와 구분합니다.

**후보 집합과 재정렬 순위**　재정렬 전 첫 후보는 마침표 하나입니다. 대출 문장을 앞으로 보내도 전체 후보 수는 같습니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter38/index.html#practice-space)


### 셀 01 · 재정렬 전 후보 순위

후보 두 건을 그대로 받아 첫 번째 청크의 본문을 읽습니다.


In [1]:
candidates = [{'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1},
 {'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1}]
print(candidates[0]['id'], repr(candidates[0]['text']))

library-1:1 '.'


마침표만 있는 library-1:1이 보입니다. 원문은 맞지만 이 조각만 받은 답변 함수에는 기간을 확인할 숫자가 없습니다.


### 셀 02 · sorted()의 key와 reverse

각 후보의 본문에 ‘대출’이 들어 있는지 검사해 해당 문장을 앞으로 보냅니다.


In [2]:
ranked = sorted(candidates, key=lambda doc: '대출' in doc['text'], reverse=True)
print(ranked[0]['id'], ranked[0]['text'])

library-1:0 도서 대출 기간은 14일입니다


‘대출’을 포함하는 library-1:0이 2위에서 1위로 올라왔습니다.

재정렬의 효과를 확인하려면 후보 목록에서 library-1:0을 잠시 빼 보세요. 남은 마침표를 아무리 정렬해도 ‘14일’은 돌아오지 않습니다. 이 셀의 점수는 ‘대출’이 들어 있는지를 나타내는 불리언 값입니다. 학습된 재정렬 모델이 질문과 문장의 뜻을 얼마나 잘 비교하는지는 별도 실험에서 측정해야 합니다.

<div class="example-table"><table><caption>같은 후보 두 건의 재정렬</caption><thead><tr><th scope="col">청크 ID</th><th scope="col">청크 본문</th><th scope="col">검색 순위</th><th scope="col">키워드 정렬 후</th></tr></thead><tbody><tr><td>library-1:1</td><td>.</td><td>1</td><td>2</td></tr><tr><td>library-1:0</td><td>도서 대출 기간은 14일입니다</td><td>2</td><td>1</td></tr></tbody></table></div>


## 질의 확장과 청크 ID 중복 제거

‘책을 빌릴 수 있는 기간은?’을 세 낱말 벡터로 바꾸면 대출 항목이 0입니다. 이번에는 검색용 표현 ‘대출 기간’을 직접 추가합니다. 원래 질문도 남겨 두고, 공백만 다른 표현은 한 번만 검색합니다.


### 과제 03 · 중복 질의 제거

원래 질문과 확장문을 순서대로 읽어 공백을 정리하고 중복 없이 남기세요.


In [3]:
queries = ['책을 빌릴 수 있는 기간은?', '대출 기간', '대출   기간']
# 첫 등장 순서를 지키며 중복을 제거하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>queries = [&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;, &#x27;대출   기간&#x27;]
unique = []
for query in queries:
    clean = &#x27; &#x27;.join(query.split())
    if clean not in unique:
        unique.append(clean)
print(unique)</code></pre>
<p>실행 결과</p><pre><code>[&#x27;책을 빌릴 수 있는 기간은?&#x27;, &#x27;대출 기간&#x27;]</code></pre>
<p>두 검색문이 남습니다. 검색문 중복을 줄였어도 양쪽 검색 결과에 같은 청크가 나올 수 있습니다. 결과를 합칠 때는 청크 ID 기준으로 한 번 더 확인합니다.</p>
</details>


### 검색어가 달라지면 후보를 다시 가져옵니다

<p>과제의 첫 질의 ‘책을 빌릴 수 있는 기간은?’은 검색 함수가 아는 단어를 포함하지 않습니다. ‘대출 기간’을 함께 보내면 첫 질의에서 빠진 문서를 찾을 수 있습니다. 셀 04에서는 두 질의 모두 ‘대출’을 포함하도록 바꿔, 같은 청크가 돌아오는 경우를 따로 실험합니다.</p>


### 무엇을 기준으로 중복을 판단할까요?

<p>요청을 보내기 전에는 공백을 정리한 검색문을 비교합니다. 결과를 합칠 때는 청크 ID를 비교합니다. 본문만 비교하면 서로 다른 쪽에 있는 같은 문장을 합쳐 출처 하나를 잃을 수 있습니다. 셀 04는 ID만 모으므로 같은 ID의 원문 충돌까지 검사하지는 않습니다.</p>


### 셀 04 · 검색 결과의 청크 ID 합집합

같은 고정 인덱스에서 검색문별 결과를 받아 첫 등장 순서대로 합칩니다. 이 예의 검색은 ‘대출’ 포함 여부로 찾는 작은 로컬 검색입니다.

과제 03에서는 ‘책을 빌릴 수 있는 기간은?’을 출발점으로 표현을 늘렸습니다. 이번 셀은 두 표현 모두 ‘대출’을 포함하는 별도 예입니다. 여기서 관찰할 것은 새 문서 발견이 아니라 두 검색이 돌려준 같은 ID를 한 번만 남기는 과정입니다.


In [4]:
index = [{'id': 'library-1:0',
  'text': '도서 대출 기간은 14일입니다',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-1:1', 'text': '.', 'source': '가상 도서관 이용 안내', 'page': 1}]
search_queries = ['대출 기간은 며칠인가요?', '대출 기간']

def find_chunks(query):
    return [row['id'] for row in index if '대출' in query and '대출' in row['text']]

batches = [find_chunks(query) for query in search_queries]
merged = list(dict.fromkeys(chunk for batch in batches for chunk in batch))
print(batches)
print(merged)

[['library-1:0'], ['library-1:0']]
['library-1:0']


질문 표현 두 개가 같은 library-1:0을 찾았습니다. 검색은 두 번 했지만 답의 근거에는 청크 하나만 넘기면 됩니다. 원래 후보에 없던 청크가 추가되려면 확장한 검색문에서 실제로 다른 결과를 얻어야 합니다.


### 과제 05 · 정답 청크가 후보에 없을 때

마침표만 남은 후보를 같은 규칙으로 재정렬하세요. 이어 원래 표현과 ‘대출 기간’을 각각 find_chunks에 넣으세요. ‘14일’이 들어 있는 청크를 되찾은 단계가 무엇인지 기록하세요.


In [ ]:
pool = candidates[:1]
# 주어진 후보만 정렬한 결과, 원래 질의의 검색 결과, 확장 질의의 검색 결과를 출력하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>pool = candidates[:1]
reranked = sorted(pool, key=lambda doc: &#x27;대출&#x27; in doc[&#x27;text&#x27;], reverse=True)
print([doc[&#x27;id&#x27;] for doc in reranked])
print(find_chunks(&#x27;책을 빌릴 수 있는 기간은?&#x27;))
print(find_chunks(&#x27;대출 기간&#x27;))</code></pre>
<p>실행 결과</p><pre><code>[&#x27;library-1:1&#x27;]
[]
[&#x27;library-1:0&#x27;]</code></pre>
<p>첫 줄은 마침표였던 후보를 그대로 반환합니다. 둘째 줄에는 ‘대출’이 없어 결과가 비었습니다. 셋째 줄은 인덱스를 다시 검색했기 때문에 후보에 없던 library-1:0을 가져왔습니다. 원래 표현과 확장 표현을 모두 실행하면 검색 함수 호출은 두 번입니다.</p>
</details>


## 돌아보기

키워드 재순위가 성공한 위 사례는 의도적으로 순서를 뒤집은 연습입니다. 실제 채택 여부는 다음 장의 같은 평가 질문에서 나온 결과와 추가 작업 횟수로 판단하세요.
